<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 DBSCAN: Multitudes, Orillas y Solitarios 🏟️
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Machine Learning
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 04
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Machine%20Learning/04%20-%20Clustering%20No%20Supervisado/Para%20Dummies/03_DBSCAN_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **"para no ingenieros"** del [cuaderno 03 (DBSCAN)](../03_DBSCAN.ipynb). *K-Means* ve cada grupo como una **zona redonda alrededor de un centro**. DBSCAN usa otra idea, mucho más natural cuando miras una plaza llena de gente: **un grupo es una multitud, es decir, un lugar donde hay mucha gente junta, separado de otros por zonas vacías**. Y la gente que está sola, lejos de todos, **no pertenece a ningún grupo**.

Al terminar podrás explicar:
1. Los dos «botones» de DBSCAN (**radio** y **mínimo de vecinos**) y los tres tipos de personas: **centro de la multitud**, **orilla** y **solitario**.
2. Por qué DBSCAN puede encontrar grupos de **formas raras** (como una media luna) donde *K-Means* falla.
3. Qué pasa si **giras mal los botones**.
4. Por qué sirve para **detectar rarezas** y cuándo **falla** (multitudes con densidades muy distintas).

> 🔗 Si luego quieres ver la versión con más detalle: [cuaderno estándar](../03_DBSCAN.ipynb). Otras miradas sencillas: [DBSCAN (Dummies) en Data Mining](../../../Data%20Mining/03%20-%20Clustering%20y%20Mineria%20Reglas%20de%20Asociacion/Para%20Dummies/02_DBSCAN_Dummies.ipynb) y [DBSCAN (Dummies) en Data Science Programming](../../../Data%20Science%20programming/10%20-%20Clustering/Para%20Dummies/04_DBSCAN_Clustering_Dummies.ipynb).

---
## 1. La plaza llena de gente 🏟️

Mira una plaza desde un balcón. Ves **grupos de personas conversando** y **algunas personas solas** caminando. Para decidir quién pertenece a un grupo, DBSCAN usa **dos botones**:

| Botón | Pregunta | Analogía |
|---|---|---|
| **Radio** (`eps`) | ¿Qué tan cerca debe estar alguien para ser «vecino»? | El alcance de **un saludo**: quien está dentro de ese círculo, es vecino |
| **Mínimo de vecinos** (`min_samples`) | ¿Cuántos vecinos hacen falta para llamarlo «multitud»? | Con menos de, digamos, 5 personas, no es una multitud |

Y cada persona cae en **uno de tres tipos**:

- 🟢 **Centro de la multitud** (*núcleo*): tiene **suficientes vecinos** dentro de su círculo. Es el corazón del grupo y **va sumando** a los que están cerca.
- 🟡 **Orilla** (*borde*): no tiene suficientes vecinos, pero está **al alcance de alguien del centro**. Pertenece al grupo, pero en la periferia.
- ⚪ **Solitario** (*ruido*): no tiene vecinos suficientes y **nadie del centro lo alcanza**. No pertenece a ningún grupo.

Un **grupo** es entonces una **cadena de centros que se alcanzan unos a otros**, más sus orillas. Lo dibujamos con datos en forma de **dos medias lunas** y unos cuantos solitarios.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN, KMeans
from sklearn.datasets import make_moons, make_blobs
from sklearn.metrics import adjusted_rand_score

rng = np.random.RandomState(1)
lunas, _ = make_moons(n_samples=260, noise=0.07, random_state=1)
solitarios = rng.uniform([-1.3, -0.8], [2.3, 1.3], size=(12, 2))             # 12 personas sueltas por la plaza
plaza = np.vstack([lunas, solitarios])

modelo = DBSCAN(eps=0.17, min_samples=5).fit(plaza)                          # radio = 0.17; multitud = al menos 5 personas (contandose a si mismo)
etiqueta = modelo.labels_
es_centro = np.zeros(len(plaza), dtype=bool); es_centro[modelo.core_sample_indices_] = True
es_orilla = (etiqueta != -1) & ~es_centro
es_solitario = etiqueta == -1

plt.figure(figsize=(9.5, 5))
plt.scatter(*plaza[es_centro].T, s=40, c=etiqueta[es_centro], cmap="coolwarm", edgecolor="black", linewidth=0.4, label="centro de la multitud")
plt.scatter(*plaza[es_orilla].T, s=55, marker="s", c=etiqueta[es_orilla], cmap="coolwarm", edgecolor="black", linewidth=0.8, label="orilla")
plt.scatter(*plaza[es_solitario].T, s=70, marker="x", color="gray", linewidth=2, label="solitario (ruido)")
plt.title("DBSCAN: centros, orillas y solitarios", fontweight="bold"); plt.legend(); plt.show()
print(f"Grupos encontrados: {etiqueta.max() + 1} | centros: {es_centro.sum()} | orillas: {es_orilla.sum()} | solitarios: {es_solitario.sum()}")
print("DBSCAN nunca preguntó cuántos grupos eran: los descubrió solo, siguiendo cadenas de gente cercana.")

---
## 2. Formas raras y botones: lo que DBSCAN puede y no puede 🌙🎛️

**Formas raras.** Como DBSCAN **sigue la gente de vecino en vecino**, puede recorrer una **media luna** o un **anillo** sin problema. *K-Means*, que solo sabe hacer zonas redondas con fronteras rectas, **corta las lunas por la mitad**. Medimos el **acuerdo** con la verdad (1 = perfecto; 0 = como adivinar).

**Los botones importan.** ¿Qué pasa si giras mal el **radio**?

- **Radio muy pequeño**: casi nadie tiene vecinos ⇒ **todos son solitarios** (o quedan grupitos diminutos).
- **Radio justo**: aparecen los grupos de verdad.
- **Radio muy grande**: todos son vecinos de todos ⇒ **la plaza entera es un solo grupo**.

Y con el **mínimo de vecinos**: si lo subes mucho, **se exige demasiada gente** para llamar multitud a algo, y más personas pasan a ser solitarias.

In [ ]:
X, real = make_moons(n_samples=400, noise=0.06, random_state=0)

# --- Parte 1: DBSCAN contra K-Means en dos medias lunas ---
con_dbscan = DBSCAN(eps=0.18, min_samples=5).fit_predict(X)
con_kmeans = KMeans(n_clusters=2, n_init=10, random_state=0).fit_predict(X)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
for ax, etiquetas, titulo in [(axes[0], con_dbscan, "DBSCAN"), (axes[1], con_kmeans, "K-Means (K = 2)")]:
    ax.scatter(X[:, 0], X[:, 1], c=etiquetas, cmap="coolwarm", s=14)
    ax.set_title(f"{titulo}\nacuerdo con lo real = {adjusted_rand_score(real, etiquetas):.2f}", fontweight="bold")
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()
print("DBSCAN recorre cada media luna de vecino en vecino; K-Means las corta con una recta.\n")

# --- Parte 2: tres radios distintos sobre las mismas lunas ---
fig, axes = plt.subplots(1, 3, figsize=(16, 4.4))
for ax, radio in zip(axes, [0.05, 0.18, 0.7]):
    etiquetas = DBSCAN(eps=radio, min_samples=5).fit_predict(X)
    n_grupos = len(set(etiquetas)) - (1 if -1 in etiquetas else 0)
    ax.scatter(X[etiquetas != -1, 0], X[etiquetas != -1, 1], c=etiquetas[etiquetas != -1], cmap="tab10", s=14)
    ax.scatter(X[etiquetas == -1, 0], X[etiquetas == -1, 1], marker="x", color="gray", s=18)
    ax.set_title(f"Radio = {radio}\n{n_grupos} grupo(s), {int((etiquetas == -1).sum())} solitarios (x gris)", fontweight="bold", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()
print("Radio muy chico: casi todos solitarios. Radio justo: dos lunas. Radio muy grande: todo se junta en un único grupo.")
print("Hay trucos para elegir el radio (el gráfico de k-distancias, en el cuaderno estándar). Y conviene poner las columnas en la misma escala antes de empezar.")

---
## 3. Los solitarios como detector de rarezas 🚨

Los «solitarios» de DBSCAN no son un fallo: son **información**. Si la mayoría de la gente forma multitudes y alguien no encaja con ninguna, **es una rareza**: un cliente atípico, una compra sospechosa, un sensor que se dañó. Probamos con tres grupos normales y **15 puntos sueltos** que añadimos a propósito. ¿Cuántos descubre DBSCAN?

In [ ]:
normales, _ = make_blobs(n_samples=500, centers=[[-4, -3], [4, -3], [0, 4]], cluster_std=0.9, random_state=42)
raros = np.random.RandomState(42).uniform(-9, 9, size=(15, 2))                  # 15 rarezas que anadimos a proposito
todos = np.vstack([normales, raros])
es_raro = np.r_[np.zeros(len(normales), bool), np.ones(len(raros), bool)]

etiquetas = DBSCAN(eps=0.8, min_samples=8).fit_predict(todos)
marcados = etiquetas == -1
print(f"DBSCAN marcó {marcados.sum()} puntos como solitarios; {(marcados & es_raro).sum()} de ellos son de las 15 rarezas que añadimos.")
print(f"Rarezas que se le escaparon: {(~marcados & es_raro).sum()} (eran puntos sueltos que cayeron casualmente cerca de un grupo).")

plt.figure(figsize=(7, 5))
plt.scatter(todos[~marcados, 0], todos[~marcados, 1], c=etiquetas[~marcados], cmap="tab10", s=12)
plt.scatter(todos[marcados, 0], todos[marcados, 1], marker="x", color="black", s=60, label="solitarios (posibles rarezas)")
plt.title("Los solitarios de DBSCAN apuntan a las rarezas", fontweight="bold"); plt.legend(); plt.show()

---
## 4. Cuándo DBSCAN se equivoca ⚠️

DBSCAN usa **un solo radio para toda la plaza**. Eso falla cuando hay **multitudes de densidades muy distintas**: una **fiesta apretada** (todos pegados) y una **reunión relajada** (la gente separada). Un radio que sirve para detectar la reunión relajada **junta las dos fiestas apretadas** que estaban cerca; y un radio pequeño, bueno para separar las fiestas apretadas, **deja a toda la reunión relajada como «solitarios»**. Lo vemos.

In [ ]:
# Dos fiestas apretadas y cercanas + una reunion relajada (mucha mas dispersion)
gente, cual = make_blobs(n_samples=[150, 150, 300], centers=[[0, 0], [0.9, 0], [5, 3]], cluster_std=[0.12, 0.12, 0.9], random_state=42)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for ax, radio in zip(axes, [0.15, 1.0]):
    etiquetas = DBSCAN(eps=radio, min_samples=10).fit_predict(gente)
    n_grupos = len(set(etiquetas)) - (1 if -1 in etiquetas else 0)
    ax.scatter(gente[etiquetas != -1, 0], gente[etiquetas != -1, 1], c=etiquetas[etiquetas != -1], cmap="tab10", s=12)
    ax.scatter(gente[etiquetas == -1, 0], gente[etiquetas == -1, 1], marker="x", color="gray", s=18)
    ax.set_title(f"Radio = {radio}: {n_grupos} grupo(s) y {int((etiquetas == -1).sum())} solitarios", fontweight="bold", fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()
print("Radio chico (izquierda): separa las dos fiestas apretadas, pero la reunión relajada queda casi toda como 'solitarios'.")
print("Radio grande (derecha): la reunión relajada es un grupo, pero las dos fiestas apretadas se funden en uno.")
print("Para estos casos existen versiones mejoradas (OPTICS y HDBSCAN), que se mencionan en el cuaderno estándar.")

---
##### 🛠️ Práctica 1: ¿Qué tan grande el radio? Contando solitarios y rarezas

Volvamos a los 3 grupos normales con **15 rarezas añadidas** (`todos`, con `es_raro` marcando cuáles son las 15). Probemos **cuatro radios** y veamos qué pasa con los solitarios:

1. Para cada radio de la lista `[0.3, 0.8, 1.5, 3.0]` ejecuta `DBSCAN(eps=radio, min_samples=8)` sobre `todos`.
2. Cuenta cuántos puntos quedan como **solitarios** (etiqueta `-1`) y cuántos de ellos son **rarezas de verdad** (`es_raro`).
3. Junta los resultados en una tabla `tabla_radios` con una fila por radio y las columnas `solitarios` y `rarezas_atrapadas`.
4. Comprueba que con el radio **más chico** hay más de 100 solitarios (¡casi todo son falsas alarmas!), que con 0.8 se atrapan al menos 10 de las 15 rarezas, y que **al agrandar el radio los solitarios nunca aumentan**.

In [ ]:
# Escribe tu código aquí

radios = [0.3, 0.8, 1.5, 3.0]

# 1) y 2) Un DBSCAN por cada radio; contar solitarios y rarezas atrapadas
# filas = []
# for radio in radios:
#     etiq = DBSCAN(eps=radio, min_samples=8).fit_predict(todos)
#     solitarios = etiq == -1
#     filas.append({"radio": radio, "solitarios": int(solitarios.sum()),
#                   "rarezas_atrapadas": int((solitarios & es_raro).sum())})

# 3) La tabla
# tabla_radios = pd.DataFrame(filas).set_index("radio")
# print(tabla_radios)

# 4) Comprobaciones
# assert tabla_radios.loc[0.3, "solitarios"] > 100
# assert tabla_radios.loc[0.8, "rarezas_atrapadas"] >= 10
# assert tabla_radios["solitarios"].is_monotonic_decreasing


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
radios = [0.3, 0.8, 1.5, 3.0]

# 1) y 2) Un DBSCAN por cada radio; contar solitarios y rarezas atrapadas
filas = []
for radio in radios:
    etiq = DBSCAN(eps=radio, min_samples=8).fit_predict(todos)
    solitarios = etiq == -1
    filas.append({"radio": radio, "solitarios": int(solitarios.sum()),
                  "rarezas_atrapadas": int((solitarios & es_raro).sum())})

# 3) La tabla
tabla_radios = pd.DataFrame(filas).set_index("radio")
print(tabla_radios)

# 4) Comprobaciones
assert tabla_radios.loc[0.3, "solitarios"] > 100
assert tabla_radios.loc[0.8, "rarezas_atrapadas"] >= 10
assert tabla_radios["solitarios"].is_monotonic_decreasing
print("\nRadio chico: demasiadas falsas alarmas (hasta la gente de las multitudes queda sola)."
      " Radio grande: casi nadie queda solo y se escapan rarezas. El radio justo (0.8) atrapa casi todas.")
```
</details>

---


---
## Resumen en una frase 🎯

> **DBSCAN forma grupos siguiendo cadenas de vecinos en zonas densas, sin que le digas cuántos grupos buscar: los centros de las multitudes crecen hacia sus orillas y los solitarios quedan marcados como ruido; recorre formas raras y detecta rarezas, pero necesita bien afinados el radio y el mínimo de vecinos, y no sabe manejar multitudes de densidades muy distintas.**

### Lo que aprendiste hoy:
- ✅ Dos botones: **radio** (qué tan cerca es «vecino») y **mínimo de vecinos** (cuántos hacen una multitud).
- ✅ Tres tipos de punto: **centro**, **orilla** y **solitario** (ruido).
- ✅ **No hay que fijar** el número de grupos y puede seguir **formas raras**.
- ✅ Un radio **muy chico** deja todo como solitarios; uno **muy grande** junta todo.
- ✅ Los **solitarios** sirven como **detector de rarezas**.
- ✅ Falla con **densidades muy distintas** (un solo radio no sirve para todas).

> 🎓 **Siguiente paso:** [Métricas de Clustering (Dummies)](04_Metricas_Clustering_Dummies.ipynb): ¿cómo sabemos si el agrupamiento quedó bien? Versión con más detalle de este tema: [cuaderno estándar](../03_DBSCAN.ipynb).

---
### 🧠 ¿Ya lo tienes claro?

1. Con tus palabras: ¿qué diferencia hay entre una persona del **centro** de una multitud, una de la **orilla** y un **solitario** en DBSCAN?
2. Un amigo dice: «con DBSCAN puse un radio enorme para no dejar a nadie como solitario, y ahora me sale un solo grupo». ¿Qué le explicarías? ¿Qué le sugerirías hacer en vez de agrandar el radio?

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Machine Learning (Edición Didáctica Para Dummies)</i>
  </p>
</div>